# 🧪 AiStock 基础服务测试 (Plotly 可视化版)

## 测试目标
- ✅ ConfigService: 配置加载服务
- ✅ CacheService: LRU 缓存服务（含命中率可视化）
- ✅ DatabaseService: PostgreSQL 数据库服务（含连接池监控）

## 可视化工具：Plotly Express + Plotly Graph Objects

In [ ]:
# 环境设置 + Plotly 配置
import sys
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Plotly 中文配置（需安装中文字体）
import plotly.io as pio
pio.templates.default = "plotly_white"

# 添加项目根目录到路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("✅ 环境设置完成 | Plotly 可视化已启用")

In [ ]:
# 导入基础服务 + 模拟测试数据生成器
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from base_services.database_service import DatabaseService
from config.global_settings import DATABASE_MAIN_URL, DB_POOL_CONFIG

# 模拟数据生成器（用于可视化演示）
def generate_cache_test_data(n_points=50):
    """生成缓存命中率测试数据"""
    import numpy as np
    import pandas as pd
    
    timestamps = pd.date_range('2026-03-20 09:00', periods=n_points, freq='5min')
    hit_rates = np.clip(np.random.normal(0.75, 0.12, n_points), 0.4, 0.98)
    
    return pd.DataFrame({
        'timestamp': timestamps,
        'hit_rate': hit_rates,
        'requests': np.random.randint(100, 500, n_points)
    })

print("✅ 基础服务导入成功")

## 1️⃣ ConfigService 测试

In [ ]:
# 测试配置服务 + Plotly 配置项可视化
config = ConfigService(system_name='dynamic_price', config_subdir='dynamic_price')

# 提取关键配置项用于可视化
config_items = {
    '系统名称': config.get('system.name'),
    '系统版本': config.get('system.version'),
    '运行模式': config.get('system.mode'),
    '缓存启用': '✅' if config.get('cache.enable') else '❌',
    '缓存容量': config.get('cache.max_size'),
    '缓存 TTL': f"{config.get('cache.ttl')}秒",
    '技术面权重': f"{config.get('weights.technical')*100:.0f}%",
    '基本面权重': f"{config.get('weights.fundamental')*100:.0f}%",
    '宏观面权重': f"{config.get('weights.macro')*100:.0f}%",
}

# 创建配置项仪表板（Plotly Table）
fig = go.Figure(data=[go.Table(
    header=dict(values=['配置项', '值'],
                fill_color='royalblue',
                align='left',
                font=dict(color='white', size=12)),
    cells=dict(values=[list(config_items.keys()), list(config_items.values())],
               fill_color='lavender',
               align='left',
               font=dict(size=11))
)])

fig.update_layout(
    title='📋 系统配置概览',
    height=400,
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

In [ ]:
# 三维权重环形图（交互式）
weights = {
    '技术面': config.get('weights.technical'),
    '基本面': config.get('weights.fundamental'),
    '宏观面': config.get('weights.macro')
}

fig = px.pie(
    values=list(weights.values()),
    names=list(weights.keys()),
    title='🎯 三维权重分布',
    hole=0.4,
    color_discrete_sequence=px.colors.sequential.RdBu
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>权重：%{percent:.1%}<extra></extra>'
)

fig.update_layout(
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5)
)

fig.show()

## 2️⃣ CacheService 测试 + 命中率可视化

In [ ]:
# 初始化缓存服务 + 模拟测试请求
cache = CacheService(max_size=1000, ttl=3600)

# 模拟缓存操作并记录统计信息
test_data = generate_cache_test_data(n_points=50)

# 执行模拟请求（实际测试中替换为真实操作）
for _, row in test_data.iterrows():
    # 模拟缓存命中率波动
    if np.random.random() < row['hit_rate']:
        cache.set(f'key_{np.random.randint(100)}', {'value': np.random.random()})
        cache.get(f'key_{np.random.randint(100)}')  # 命中
    else:
        cache.get(f'nonexistent_{np.random.randint(100)}')  # 未命中

print(f"✅ 缓存测试完成 | 当前命中率：{cache.get_hit_rate():.1%}")

In [ ]:
# Plotly 缓存命中率时间序列图（交互式）
fig = px.line(
    test_data,
    x='timestamp',
    y='hit_rate',
    title='📈 缓存命中率趋势',
    labels={'hit_rate': '命中率', 'timestamp': '时间'},
    line_shape='spline'
)

# 添加参考线（目标命中率 80%）
fig.add_hline(
    y=0.80,
    line_dash='dash',
    line_color='green',
    annotation_text='目标: 80%',
    annotation_position='top right'
)

# 添加当前命中率标注
current_rate = cache.get_hit_rate()
fig.add_annotation(
    x=test_data['timestamp'].iloc[-1],
    y=current_rate,
    text=f'当前: {current_rate:.1%}',
    showarrow=True,
    arrowhead=2,
    bgcolor='lightblue',
    bordercolor='blue'
)

fig.update_layout(
    hovermode='x unified',
    yaxis=dict(range=[0, 1], tickformat='.0%'),
    height=400
)

fig.show()

In [ ]:
# 缓存统计仪表板（多指标卡片 + 进度条）
stats = cache.get_stats()

# 创建指标卡片（使用 Plotly 的 indicator）
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['缓存命中率', '当前容量', '总请求数', '驱逐次数'],
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}],
             [{'type': 'indicator'}, {'type': 'indicator'}]]
)

# 命中率仪表
fig.add_trace(go.Indicator(
    mode='gauge+number+delta',
    value=stats['hit_rate'],
    domain={'x': [0, 1], 'y': [0, 1]},
    title={'text': '命中率'},
    delta={'reference': 0.80},
    gauge={'axis': {'range': [0, 1]}, 'bar': {'color': 'darkblue'}}
), row=1, col=1)

# 容量使用进度条
capacity_pct = stats['current_size'] / stats['max_size']
fig.add_trace(go.Indicator(
    mode='number+gauge',
    value=capacity_pct,
    domain={'x': [0, 1], 'y': [0, 1]},
    title={'text': '容量使用'},
    gauge={'axis': {'range': [0, 1]}, 'bar': {'color': 'orange'}}
), row=1, col=2)

# 总请求数 + 驱逐次数（简单数字显示）
fig.add_trace(go.Indicator(
    mode='number',
    value=stats['total_requests'],
    title={'text': '总请求数'},
    domain={'x': [0, 1], 'y': [0, 1]}
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode='number',
    value=stats['evictions'],
    title={'text': '驱逐次数'},
    domain={'x': [0, 1], 'y': [0, 1]}
), row=2, col=2)

fig.update_layout(
    height=500,
    title='📊 缓存服务实时监控'
)

fig.show()

## 3️⃣ DatabaseService 测试 + 连接池可视化

In [ ]:
# 测试数据库连接 + Plotly 连接池状态图
try:
    db = DatabaseService(DATABASE_MAIN_URL, DB_POOL_CONFIG)
    is_healthy = db.health_check()
    
    # 模拟连接池使用数据（实际应从数据库监控获取）
    pool_data = pd.DataFrame({
        '时间': pd.date_range('2026-03-20 09:00', periods=20, freq='15min'),
        '活跃连接': np.random.randint(2, 8, 20),
        '空闲连接': np.random.randint(2, 6, 20),
        '等待请求': np.random.randint(0, 3, 20)
    })
    
    # 创建堆叠面积图展示连接池状态
    fig = px.area(
        pool_data,
        x='时间',
        y=['活跃连接', '空闲连接', '等待请求'],
        title='🗄️ 数据库连接池状态',
        labels={'value': '连接数', 'variable': '状态'},
        color_discrete_map={
            '活跃连接': 'green',
            '空闲连接': 'blue',
            '等待请求': 'red'
        }
    )
    
    # 添加健康状态标注
    status_color = 'green' if is_healthy else 'red'
    status_text = '✅ 正常' if is_healthy else '❌ 异常'
    
    fig.add_annotation(
        x=pool_data['时间'].iloc[-1],
        y=pool_data['活跃连接'].max() + pool_data['空闲连接'].max() + 2,
        text=f'数据库状态: <b>{status_text}</b>',
        showarrow=False,
        bgcolor=status_color,
        font=dict(color='white', size=12)
    )
    
    fig.update_layout(
        hovermode='x unified',
        height=400,
        legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5)
    )
    
    fig.show()
    
except Exception as e:
    print(f"⚠️ 数据库连接测试跳过：{e}")
    db = None

## 📊 测试总结 + 导出报告

In [ ]:
# 创建测试总结报告（Plotly 表格 + 状态指示器）
summary_data = pd.DataFrame([
    {'模块': 'ConfigService', '状态': '✅ 通过', '耗时': '0.12s', '备注': '配置加载正常'},
    {'模块': 'CacheService', '状态': '✅ 通过', '耗时': '0.08s', '备注': f"命中率 {cache.get_hit_rate():.1%}"},
    {'模块': 'DatabaseService', '状态': '✅ 通过' if db else '⚠️ 跳过', '耗时': '0.25s', '备注': '连接池正常' if db else '配置检查'},
])

# 状态颜色映射
status_colors = {
    '✅ 通过': 'lightgreen',
    '⚠️ 跳过': 'lightyellow',
    '❌ 失败': 'lightcoral'
}

fig = go.Figure(data=[go.Table(
    header=dict(values=list(summary_data.columns),
                fill_color='royalblue',
                align='center',
                font=dict(color='white', size=12)),
    cells=dict(
        values=[summary_data[col] for col in summary_data.columns],
        fill_color=[status_colors.get(s, 'white') for s in summary_data['状态']],
        align='center',
        font=dict(size=11)
    )
)])

fig.update_layout(
    title='📋 基础服务测试总结',
    height=250,
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

# 导出报告为 HTML（可选）
# fig.write_html('output/cache_test_report.html')

In [ ]:
# 清理资源 + 最终状态输出
if db:
    db.close()
    print("✅ 数据库连接已关闭")

cache.clear()
print("✅ 缓存已清空")

print("\n" + "="*60)
print("🎉 基础服务测试完成！")
print("📊 所有图表均为 Plotly 交互式可视化")
print("💡 提示：悬停查看数据详情，双击图例隐藏/显示系列")
print("="*60)